# 14 — Post-rung-4 diagnosis: score the honest CNN ensemble, check the CV-vs-leaderboard gap

**Why this notebook exists**: an Opus strategic review (2026-09-10, after rung 4
concluded 0/5) found that **no gate in this project has ever scored the thing
that actually ships**. Rung 3's gate scores the mean log loss of 5 *individual*
CNN OOF arrays (`rung3_oof_seed42..46.npy`, mean 0.4520). Production
(`submission_src/main.py`) averages **25 checkpoints'** predictions per row,
then blends with the classical baseline at `w_cnn=0.70`. The average of 5
models' scores is not the score of the average of 5 models' predictions.

**The fix is free**: fold splits are a deterministic function of
`(labels, families, random_state=seed)`, so for a given row, `rung3_oof_seed{s}`
already holds the prediction of the one checkpoint (seed `s`) that never saw
that row during training. `mean(rung3_oof_seed42..46)` is therefore a
legitimate — and conservative, since production averages 25 checkpoints not 5
— out-of-fold estimate of the shipped CNN ensemble. Nobody has scored it.

**What this notebook does** (CPU-only, no GPU, no volume cache, milliseconds —
just loads existing `.npy`/`.csv` files already on disk):
1. Scores the honest 5-way CNN-ensemble OOF: log loss, AUROC, ECE.
2. Scores the existing blend (ensemble CNN + classical baseline at the current
   `w_cnn=0.70`) the same way.
3. Compares both against the real leaderboard numbers from the first full
   submission (log loss 0.4648, AUROC 0.8796) to check whether the
   CV-vs-leaderboard gap is **discrimination** (AUROC also drops -> real
   distribution shift) or **calibration** (AUROC matches but log loss doesn't
   -> a one-parameter recalibration fix, not yet tried anywhere in this
   pipeline, could recover most of the gap).

**Data handling**: loads real row-level labels and OOF prediction arrays, so
per the AI-assistant data rule (`README.md`) this is **[RUN ME]** — run it
yourself, share back only the printed aggregate numbers, never any per-row
output. (No GPU/volume access needed — this is the cheapest possible
[RUN ME] cell in the project.)

**Source**: `project_dat_parkinson_strategic_roadmap.md` project memory
(this session), item 1 of the ranked roadmap.

In [1]:
# [RUN ME] -- loads real row-level labels + existing OOF prediction arrays.
# CPU-only, no GPU, no volume cache -- self-contained like notebook 07's own
# LOFO cell, does not assume any earlier cell ran in this kernel session.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd

import config
import evaluate

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)
labels = labeled_df[config.TARGET_COLUMN].tolist()
y_true = np.array(labels)

repeat_seeds = list(range(config.SEED, config.SEED + 5))
cnn_oof_repeats = [np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds]
baseline_oof = np.load(config.DATA_PROCESSED / "baseline_oof_seed_match.npy")

W_CNN = 0.70  # current production blend weight, README.md 2026-09-09

cnn_ensemble_oof = np.mean(cnn_oof_repeats, axis=0)
blend_ensemble_oof = W_CNN * cnn_ensemble_oof + (1 - W_CNN) * baseline_oof

print(f"{len(y_true)} labeled rows, {len(cnn_oof_repeats)} CNN repeats averaged\n")

1362 labeled rows, 5 CNN repeats averaged



In [2]:
# [RUN ME] (continues from the cell above -- no new data access itself).
# Score the honest ensemble two ways, and put them next to the real
# leaderboard numbers from the first full submission.
def report(name, probs):
    scores = evaluate.combined_score(y_true, probs)
    print(f"{name:32s} log loss={scores['log_loss']:.4f}  "
          f"AUROC={scores['auroc']:.4f}  ECE={scores['ece']:.4f}")
    return scores

print("--- never-before-scored quantities ---")
cnn_ensemble_scores = report("CNN ensemble (5-way OOF)", cnn_ensemble_oof)
blend_ensemble_scores = report("Blend (ensemble CNN + baseline)", blend_ensemble_oof)

print("\n--- for reference, numbers already on record ---")
print(f"{'CNN single-repeat mean (gate)':32s} log loss=0.4520  (sd 0.0109, README.md 2026-09-09)")
print(f"{'Blend single-repeat LOFO mean':32s} log loss=0.4250  (sd 0.0099, README.md 2026-09-09)")
print(f"{'Real leaderboard (first submission)':32s} log loss=0.4648  AUROC=0.8796")

print("\n--- gap diagnosis ---")
auroc_gap = 0.8796 - blend_ensemble_scores["auroc"]
logloss_gap = 0.4648 - blend_ensemble_scores["log_loss"]
print(f"blend-ensemble OOF vs. real leaderboard: "
      f"AUROC gap={auroc_gap:+.4f}, log loss gap={logloss_gap:+.4f}")
print("If the AUROC gap is small/near-zero while the log loss gap is large: "
      "discrimination transferred, the gap is calibration -- a temperature/"
      "Platt fix (never tried in this pipeline) is the next step.")
print("If the AUROC gap is large too: the model discriminates worse on the "
      "real test distribution -- a real shift, and shrinkage toward the base "
      "rate is the more appropriate response than sharpening.")

--- never-before-scored quantities ---
CNN ensemble (5-way OOF)         log loss=0.4127  AUROC=0.8915  ECE=0.0316
Blend (ensemble CNN + baseline)  log loss=0.4119  AUROC=0.9081  ECE=0.0747

--- for reference, numbers already on record ---
CNN single-repeat mean (gate)    log loss=0.4520  (sd 0.0109, README.md 2026-09-09)
Blend single-repeat LOFO mean    log loss=0.4250  (sd 0.0099, README.md 2026-09-09)
Real leaderboard (first submission) log loss=0.4648  AUROC=0.8796

--- gap diagnosis ---
blend-ensemble OOF vs. real leaderboard: AUROC gap=-0.0285, log loss gap=+0.0529
If the AUROC gap is small/near-zero while the log loss gap is large: discrimination transferred, the gap is calibration -- a temperature/Platt fix (never tried in this pipeline) is the next step.
If the AUROC gap is large too: the model discriminates worse on the real test distribution -- a real shift, and shrinkage toward the base rate is the more appropriate response than sharpening.


**What we're looking for:** has the honestly-ensembled CNN (and the blend
built on it) actually been scored before? Does the CV-vs-leaderboard gap look
like a calibration problem (AUROC transfers, log loss doesn't) or a real
distribution-shift problem (AUROC drops too)?

**What we found:** *(paste: the CNN-ensemble and blend-ensemble log
loss/AUROC/ECE lines; the AUROC gap and log loss gap vs. the real
leaderboard numbers)*

**Decision / next step:** *(if the AUROC gap is small and the log loss gap is
large: proceed to a calibration notebook -- fit temperature/Platt scaling via
leave-one-repeat-out on this ensemble, same discipline as notebook 07's
blend-weight LOFO cell -- item 2 of the roadmap. If the AUROC gap is also
large: the priority shifts to diagnosing *why* discrimination doesn't
transfer (distribution shift) before trying to fix calibration on top of it.
Either way: if `cnn_ensemble_scores`/`blend_ensemble_scores` already beat
0.4520/0.4250 meaningfully, that alone is worth folding into the next
submission's inference code (`submission_src/main.py` already averages the
25 checkpoints the same way -- this just confirms/quantifies what that
buys). See `project_dat_parkinson_strategic_roadmap.md` for the full ranked
roadmap this notebook is item 1 of.)*